# Biblical Entity Force Graph
Builds an interactive force graph from raindrop.io Wikipedia bookmarks,
using the Wikipedia PHP API to discover internal link relationships.

In [1]:
import json
import time
import re
from pathlib import Path
from urllib.parse import urlparse, unquote

import pandas as pd
import requests
from cosmograph import cosmo

## 1. Load Raindrop Export

In [2]:
df = pd.read_csv("raindrop-export-2026-04-02/export.csv")

def extract_wiki_title(url: str) -> str | None:
    """Extract clean article title from a Wikipedia URL."""
    try:
        parsed = urlparse(url)
        if "wikipedia.org" not in parsed.netloc:
            return None
        path = parsed.path  # e.g. /wiki/Dispensationalism
        match = re.match(r"/wiki/(.+)", path)
        if not match:
            return None
        title = unquote(match.group(1)).replace("_", " ").split("#")[0].strip()
        return title if title else None
    except Exception:
        return None

df["wiki_title"] = df["url"].apply(extract_wiki_title)
wiki_articles = df[df["wiki_title"].notna()][["wiki_title", "title", "url"]].drop_duplicates("wiki_title")

print(f"Total bookmarks: {len(df)}")
print(f"Wikipedia articles: {len(wiki_articles)}")
wiki_articles[["wiki_title", "title"]].head(10)

Total bookmarks: 66
Wikipedia articles: 60


,wiki_title,title
0,Dispensationalism,Dispensationalism - Wikipedia
3,Gary North (economist),https://en.wikipedia.org/wiki/Gary_North_(econ...
4,Christian reconstructionism,https://en.wikipedia.org/wiki/Christian_recons...
5,The Late Great Planet Earth,https://en.wikipedia.org/wiki/The_Late_Great_P...
6,Hal Lindsey,https://en.wikipedia.org/wiki/Hal_Lindsey?wpro...
7,Biblical numerology,https://en.wikipedia.org/wiki/Biblical_numerol...
8,Edgar C. Whisenant,https://en.wikipedia.org/wiki/Edgar_C._Whisena...
9,Kenosis,https://en.wikipedia.org/wiki/Kenosis?wprov=sf...
10,Hermitage (religious retreat),https://en.wikipedia.org/wiki/Hermitage_(relig...
11,British New Church Movement,https://en.wikipedia.org/wiki/British_New_Chur...


## 2. Fetch Internal Links via Wikipedia PHP API
For each article, fetch which other articles in our set it links to.

In [3]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
CACHE_FILE = Path(".wiki_links_cache.json")

cache: dict = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

def fetch_links(title: str) -> list[str]:
    """Fetch all internal links from a Wikipedia article (cached)."""
    if title in cache:
        return cache[title]

    links = []
    params = {
        "action": "query",
        "prop": "links",
        "titles": title,
        "pllimit": "max",
        "format": "json",
        "plnamespace": 0,
    }
    while True:
        r = requests.get(WIKI_API, params=params, headers={"User-Agent": "biblical-graph/1.0"})
        data = r.json()
        pages = data.get("query", {}).get("pages", {})
        for page in pages.values():
            for link in page.get("links", []):
                links.append(link["title"])
        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.1)

    cache[title] = links
    CACHE_FILE.write_text(json.dumps(cache, indent=2))
    return links

article_titles = set(wiki_articles["wiki_title"].tolist())
print(f"Fetching links for {len(article_titles)} articles...")

for i, title in enumerate(article_titles):
    fetch_links(title)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(article_titles)}")

print("Done.")

Fetching links for 60 articles...
  10/60
  20/60
  30/60
  40/60
  50/60
  60/60
Done.


## 3. Build Nodes & Edges

In [4]:
# Nodes: one per Wikipedia article in our set
nodes = wiki_articles.rename(columns={"wiki_title": "id"})

# Edges: link source -> target only if target is also in our set
edges = []
for title in article_titles:
    for linked in cache.get(title, []):
        if linked in article_titles and linked != title:
            edges.append({"source": title, "target": linked})

edges_df = pd.DataFrame(edges).drop_duplicates()

print(f"Nodes: {len(nodes)}")
print(f"Edges: {len(edges_df)}")
edges_df.head()

Nodes: 60
Edges: 194


,source,target
0,Masoretic Text,Diatessaron
1,Masoretic Text,Leningrad Codex
2,Masoretic Text,Septuagint
3,George Inness,Emanuel Swedenborg
4,George Inness,Luminism (American art style)


## 4. Render Force Graph

In [5]:
cosmo(
    points=nodes,
    links=edges_df,
    point_id_by="id",
    point_label_by="id",
    link_source_by="source",
    link_target_by="target",
    simulation_gravity=0.1,
    simulation_repulsion=2.0,
    simulation_link_spring=1.0,
    simulation_friction=0.85,
    show_labels=True,
)